# Demo Causal Transformer (Dựa trên Chương 8 - NLP Textbook)
**Mục tiêu:** Xây dựng một mô hình ngôn ngữ Autoregressive (Decoder-only Transformer) từ đầu bằng PyTorch để sinh văn bản.

**Dataset:** [Netflix Movies and TV Shows](https://www.kaggle.com/datasets/shivamb/netflix-shows) từ Kaggle. Chúng ta sẽ dùng cột `description` để huấn luyện mô hình tự sáng tác cốt truyện phim Netflix.

**Các phần được cài đặt:**
1. Khởi tạo Input & Positional Embeddings 
2. Scaled Dot-Product Attention & Causal Masking 
3. Multi-Head Attention 
4. Transformer Block với Pre-LayerNorm và Residual Connection 
5. Language Modeling Head 
6. Huấn luyện và Top-k Sampling 

In [1]:
!pip install -q kaggle

import os
import torch
import torch.nn as nn
from torch.nn import functional as F
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv('/kaggle/input/datasets/shivamb/netflix-shows/netflix_titles.csv')
text_data = " ".join(df['description'].dropna().tolist())

print(f"Tổng số ký tự trong dataset: {len(text_data)}")
print(f"Ví dụ một đoạn văn bản: \n{text_data[:200]}")

Tổng số ký tự trong dataset: 1270878
Ví dụ một đoạn văn bản: 
As her father nears the end of his life, filmmaker Kirsten Johnson stages his death in inventive and comical ways to help them both face the inevitable. After crossing paths at a party, a Cape Town te


In [2]:
# Để đơn giản và trực quan với ma trận E (Embedding) trong mục 8.4, 
# ta dùng Tokenization ở mức ký tự (Character-level).
chars = sorted(list(set(text_data)))
vocab_size = len(chars)
print(f"Kích thước từ vựng (Vocabulary size |V|): {vocab_size}")

# Mapping string <-> integer
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

# Chuyển toàn bộ text thành tensor
data = torch.tensor(encode(text_data), dtype=torch.long)

# Chia Train/Val split
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

# Thiết lập siêu tham số (Hyperparameters)
batch_size = 64
block_size = 128     # Context window N (Độ dài chuỗi đầu vào)
max_iters = 3000
learning_rate = 3e-4
eval_interval = 300
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Architecture params
d_model = 128       # Chiều không gian của embedding (d)
n_heads = 4         # Số lượng attention heads (A)
n_layers = 4        # Số lượng Transformer Blocks

Kích thước từ vựng (Vocabulary size |V|): 128


In [3]:
def get_batch(split):
    # Tạo batch input X và target Y (Y dịch sang phải 1 token so với X)
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(200)
        for k in range(200):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

### Causal Self-Attention
Cài đặt một `Head` tính toán self-attention có masking:
$Q = XW^Q, K = XW^K, V = XW^V$
$head = \text{softmax}\left(\text{mask}\left(\frac{QK^T}{\sqrt{d_k}}\right)\right) V$
Masking (ma trận tam giác trên gán bằng $-\infty$) đảm bảo tính left-to-right.

In [4]:
class Head(nn.Module):
    """ Một head của self-attention (Mục 8.1.1) """
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(d_model, head_size, bias=False)
        self.query = nn.Linear(d_model, head_size, bias=False)
        self.value = nn.Linear(d_model, head_size, bias=False)
        # Tạo Mask (Mục 8.3 - Masking out the future)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)   # (B, T, head_size)
        q = self.query(x) # (B, T, head_size)
        
        # Tính attention scores: (q * k^T) / sqrt(d_k)
        wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5
        # Áp dụng causal mask
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        
        v = self.value(x) # (B, T, head_size)
        out = wei @ v     # (B, T, head_size)
        return out

class MultiHeadAttention(nn.Module):
    """ Nhiều attention heads chạy song song và nối lại (Mục 8.1.1 & 8.3) """
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, d_model) # W^O

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        return out

class FeedForward(nn.Module):
    """ Lớp Feedforward 2 lớp (Mục 8.2 Eq. 8.21) """
    def __init__(self, d_model):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.ReLU(),
            nn.Linear(4 * d_model, d_model), # Chiều chiếu ngược lại residual stream
        )

    def forward(self, x):
        return self.net(x)

class TransformerBlock(nn.Module):
    """ Transformer Block với Residual Stream và Pre-Norm (Mục 8.2 Hình 8.7) """
    def __init__(self, d_model, n_heads):
        super().__init__()
        head_size = d_model // n_heads
        self.sa = MultiHeadAttention(n_heads, head_size)
        self.ffwd = FeedForward(d_model)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x):
        # Eq 8.26 - 8.28: x = x + Attention(LayerNorm(x))
        x = x + self.sa(self.ln1(x))
        # Eq 8.29 - 8.31: x = x + FFN(LayerNorm(x))
        x = x + self.ffwd(self.ln2(x))
        return x

### Language Modeling (The Big Picture)
Ghép nối Token Embeddings ($E$), Positional Embeddings ($P$), các Blocks và Language Modeling Head (Linear layer unembedding).

In [5]:
class AutoregressiveTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        # Token Embedding Matrix E (Mục 8.4)
        self.token_embedding_table = nn.Embedding(vocab_size, d_model)
        # Absolute Positional Embedding (Mục 8.4)
        self.position_embedding_table = nn.Embedding(block_size, d_model)
        
        # N Transformer Blocks
        self.blocks = nn.Sequential(*[TransformerBlock(d_model, n_heads=n_heads) for _ in range(n_layers)])
        
        # Lớp Norm cuối cùng
        self.ln_f = nn.LayerNorm(d_model)
        
        # Language Modeling Head - Unembedding layer (Mục 8.5 Eq. 8.46)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        # Lấy embeddings
        tok_emb = self.token_embedding_table(idx) # (B, T, d_model)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T, d_model)
        
        # X = Word Embeddings + Positional Embeddings
        x = tok_emb + pos_emb # (B, T, d_model)
        
        # Đưa qua các blocks
        x = self.blocks(x)
        x = self.ln_f(x)
        
        # Lấy logits (u)
        logits = self.lm_head(x) # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits_view = logits.view(B*T, C)
            targets_view = targets.view(B*T)
            # Cross-entropy loss (Mục 8.7)
            loss = F.cross_entropy(logits_view, targets_view)

        return logits, loss

    # Mục 8.6.1: Top-k Sampling Generation
    def generate(self, idx, max_new_tokens, top_k=10):
        for _ in range(max_new_tokens):
            # Cắt idx để không vượt quá block_size
            idx_cond = idx[:, -block_size:]
            
            # Dự đoán
            logits, _ = self(idx_cond)
            # Chỉ lấy logits của token cuối cùng
            logits = logits[:, -1, :] # (B, C)
            
            # Top-k Sampling
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
                
            # Đưa qua softmax để thành xác suất (Eq 8.47)
            probs = F.softmax(logits, dim=-1)
            
            # Sample từ phân phối xác suất
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            
            # Nối vào chuỗi hiện tại
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = AutoregressiveTransformer().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
print(f"Khởi tạo xong mô hình với {sum(p.numel() for p in model.parameters())} tham số.")

Khởi tạo xong mô hình với 841088 tham số.


In [6]:
print("Bắt đầu huấn luyện...")
for iter in range(max_iters):
    # Đánh giá sau mỗi khoảng eval_interval
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss(model)
        print(f"Bước {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # Lấy batch
    xb, yb = get_batch('train')

    # Forward pass
    logits, loss = model(xb, yb)
    
    # Backward pass & update weights
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    
print("Hoàn thành huấn luyện!")

Bắt đầu huấn luyện...
Bước 0: train loss 4.9758, val loss 4.9756
Bước 300: train loss 2.5134, val loss 2.5084
Bước 600: train loss 2.3717, val loss 2.3693
Bước 900: train loss 2.1355, val loss 2.1394
Bước 1200: train loss 1.9958, val loss 2.0008
Bước 1500: train loss 1.8930, val loss 1.9023
Bước 1800: train loss 1.8032, val loss 1.8168
Bước 2100: train loss 1.7338, val loss 1.7532
Bước 2400: train loss 1.6765, val loss 1.6971
Bước 2700: train loss 1.6273, val loss 1.6544
Bước 2999: train loss 1.5904, val loss 1.6200
Hoàn thành huấn luyện!


In [7]:
prompt = "In a dark city,"
context = torch.tensor((encode(prompt)), dtype=torch.long, device=device).unsqueeze(0)

print("--- KẾT QUẢ MÔ TẢ PHIM ĐƯỢC SINH RA (AI GENERATED NETFLIX SYNOPSIS) ---")
generated_indices = model.generate(context, max_new_tokens=400, top_k=5)[0].tolist()
print(decode(generated_indices))

--- KẾT QUẢ MÔ TẢ PHIM ĐƯỢC SINH RA (AI GENERATED NETFLIX SYNOPSIS) ---
In a dark city, a crosiced who his four home own tourn while when he milliage and mission. In a can on operievers and a start attacked by high to stolen and as a then only street them an trains the dramate into their does and and true. Whreas stranged with a sent-set-up seconding someths for the count of storial and secrets at tens on the spirilt around the pass. An aspiring in to altrapplores and a secret in th
